# Task 2: Reproduce the BTCV Segmentation Results

This notebook runs inference on the 6 BTCV validation CT volumes (`img0035` – `img0040`) using the three
official pretrained Swin UNETR model configurations:
- **Swin UNETR / Base** (`feature_size=48`, 62.1M params, SSL pre-trained)
- **Swin UNETR / Small** (`feature_size=24`, 15.7M params)
- **Swin UNETR / Tiny** (`feature_size=12`, 4.0M params)

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

# Add the repository root to sys.path so `from src...` works inside notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, DATASET_JSON_PATH, METRICS_DIR, MODEL_CONFIGS
from src.data import get_validation_loader
from src.download import download_dataset_json, download_pretrained_weights
from src.inference import get_device, run_btcv_inference

device = get_device()
print(f"Project root: {PROJECT_ROOT}")
print(f"Using compute device: {device}")

Project root: /Users/paula/Desktop/Foundations of AI/Project_1
Using compute device: cpu


In [2]:
# Download official MONAI BTCV split file (dataset_0.json)
json_path = download_dataset_json()

# Download all 3 pretrained model checkpoints
ckpt_paths = download_pretrained_weights(model_keys=("base", "small", "tiny"))


[SKIP] Already exists: dataset_0.json
[SKIP] Already exists: swin_unetr.base_5000ep_f48_lr2e-4_pretrained.pt
[SKIP] Already exists: swin_unetr.small_5000ep_f24_lr2e-4_pretrained.pt
[SKIP] Already exists: swin_unetr.tiny_5000ep_f12_lr2e-4_pretrained.pt


In [3]:
with open(DATASET_JSON_PATH, "r") as f:
    split_meta = json.load(f)

train_imgs = {item["image"] for item in split_meta["training"]}
val_imgs = {item["image"] for item in split_meta["validation"]}
assert len(train_imgs & val_imgs) == 0, "Leakage detected!"
print(f"[OK] Zero split leakage: {len(train_imgs)} train scans and {len(val_imgs)} val scans are disjoint.\n")

# Verify that the 6 validation CT images and labels exist in data/btcv/
missing_files = []
for item in split_meta["validation"]:
    for key in ("image", "label"):
        fpath = DATA_DIR / item[key]
        if not fpath.exists():
            missing_files.append(str(fpath.relative_to(PROJECT_ROOT)))

if missing_files:
    print("[WARNING] The following BTCV validation NIfTI files are not yet in data/btcv/:")
    for m in missing_files:
        print(f"  - {m}")
else:
    print("[OK] All 6 BTCV validation CT images and ground-truth labels are present!")
    val_loader, val_files = get_validation_loader()

[OK] Zero split leakage: 24 train scans and 6 val scans are disjoint.

[OK] All 6 BTCV validation CT images and ground-truth labels are present!


In [4]:
INFER_OVERLAP = 0.5
SW_BATCH_SIZE = 4 if device.type == "cuda" else 1 # Lower batch size on CPU to save RAM

all_results = {}
METRICS_DIR.mkdir(parents=True, exist_ok=True)

for model_key in ("tiny", "small", "base"):
    res = run_btcv_inference(
        model_key=model_key,
        val_loader=val_loader,
        device=device,
        infer_overlap=INFER_OVERLAP,
        sw_batch_size=SW_BATCH_SIZE,
        save_predictions=True,
    )
    all_results[model_key] = res

    # Save per-case & per-organ Dice table for this model to CSV (can be used in Task 3 & Task 5)
    df_cases = pd.DataFrame(res["cases"])
    csv_path = METRICS_DIR / f"btcv_dice_{model_key}_overlap{INFER_OVERLAP}.csv"
    df_cases.to_csv(csv_path, index=False)
    print(f"Saved metrics CSV to: {csv_path.relative_to(PROJECT_ROOT)}\n")

Inference [Swin UNETR / Tiny]:  17%|█▋        | 1/6 [03:05<15:28, 185.72s/it]

  img0035.nii.gz: Mean Dice = 64.18%


Inference [Swin UNETR / Tiny]:  33%|███▎      | 2/6 [06:06<12:10, 182.72s/it]

  img0036.nii.gz: Mean Dice = 66.89%


Inference [Swin UNETR / Tiny]:  50%|█████     | 3/6 [08:51<08:43, 174.64s/it]

  img0037.nii.gz: Mean Dice = 67.72%


... (26 outputs left out)

Inference [Swin UNETR / Base]:  83%|████████▎ | 5/6 [1:04:30<12:54, 774.28s/it]

  img0039.nii.gz: Mean Dice = 83.82%


Inference [Swin UNETR / Base]: 100%|██████████| 6/6 [1:20:37<00:00, 806.25s/it]

  img0040.nii.gz: Mean Dice = 82.88%
==> [Swin UNETR / Base] Overall Mean Dice: 81.37%
Saved metrics CSV to: results/metrics/btcv_dice_base_overlap0.5.csv



In [11]:
summary_rows = []
for model_key, res in all_results.items():
    cfg = MODEL_CONFIGS[model_key]
    reproduced_dice = res["overall_mean_dice"] * 100.0
    paper_dice = cfg["paper_dice_05"] if INFER_OVERLAP == 0.5 else cfg["paper_dice_07"]
    summary_rows.append({
        "Model": cfg["name"],
        "Feature Size": cfg["feature_size"],
        "Params (M)": cfg["params_m"],
        "SSL Pre-trained": cfg["ssl_pretrained"],
        f"Paper Dice (overlap={INFER_OVERLAP}) [%]": paper_dice,
        f"Reproduced Dice (overlap={INFER_OVERLAP}) [%]": round(reproduced_dice, 2),
        "Difference [%]": round(reproduced_dice - paper_dice, 2),
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(METRICS_DIR / f"btcv_summary_overlap{INFER_OVERLAP}.csv", index=False)
df_summary

,Model,Feature Size,Params (M),SSL Pre-trained,Paper Dice (overlap=0.5) [%],Reproduced Dice (overlap=0.5) [%],Difference [%]
0,Swin UNETR / Tiny,12,4.0,False,70.35,66.31,-4.04
1,Swin UNETR / Small,24,15.7,False,79.34,77.89,-1.45
2,Swin UNETR / Base,48,62.1,True,81.86,81.37,-0.49
